# vigilex – Notebook 02
## Recall Labels: FDA Recall API × MAUDE

**Ziel:**
- Recalls für unsere Gerätekategorien aus der FDA Recall API ziehen
- MAUDE-Records mit Recall-Events matchen
- Zwei Label-Strategien: Device-Level und Time-Window
- Ergebnis als Parquet speichern: `data/processed/maude_labeled.parquet`

**Match-Logik:**
MAUDE hat `product_code` (LZG, PKU, OYC) und `manufacturer_name`.
Recall DB hat `product_code` und `recalling_firm`.
Primary Match: `product_code` + fuzzy `manufacturer_name`.

**Zwei Labels:**
- `recalled_ever`: Hat dieses Gerät (brand + manufacturer) jemals einen Recall gehabt? (einfach, für Baseline)
- `recalled_within_12m`: Gab es innerhalb von 12 Monaten nach dem Report einen Recall für dieses Gerät? (realistischer, für finales Modell)

---
## 📖 How to Read This Notebook (Beginner Guide)

This is **Notebook 02** — the "label building" step.
In machine learning, a **label** (also called "target" or "ground truth") is the
answer we want the model to learn to predict. Here the question is:
> *"Did this device eventually get recalled by the FDA?"*

### Key concepts explained here

**FDA Recall**
: When a medical device is found to be defective or dangerous, the FDA can order
  a recall. Recalls are classified by severity:
  - **Class I**: serious risk of health harm or death
  - **Class II**: temporary or reversible health harm
  - **Class III**: unlikely to cause harm (labelling issues etc.)
  In SentinelAI, the goal is to detect early signals that PREDICT a future recall.

**Label** (machine learning)
: The "correct answer" column in your training data. Our labels:
  - `recalled_ever` (boolean): Was this device model ever recalled?
  - `recalled_within_12m` (boolean): Was it recalled within 12 months of this report?

**Fuzzy matching**
: Manufacturer names in two different databases are often not identical strings
  ("Medtronic Inc." vs "MEDTRONIC" vs "Medtronic, Inc."). Fuzzy matching finds
  likely matches even when strings are not exactly equal, using edit distance or
  normalisation (lowercasing, removing punctuation, etc.).

**Time-window label**
: More realistic than `recalled_ever` for signal detection. Asking "was there a
  recall within 12 months AFTER this report was filed?" simulates a real-world
  early warning scenario. This avoids data leakage (using future information
  to predict the past).

**Data leakage**
: Accidentally using information that would not be available at prediction time.
  Example: using the recall date itself as a feature. If we train on that, the
  model "cheats" and learns to predict trivially.

### What this notebook produces
- `data/processed/maude_labeled.parquet` — MAUDE records with recall labels attached
- Understanding of label distribution and class imbalance

### Note: This notebook has not been run yet
It requires the MAUDE data from Notebook 01 and the FDA Recall API.
It will be run as part of Module 3 validation (retrospective signal analysis).
---

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path
from dotenv import load_dotenv
import os
from rapidfuzz import fuzz

load_dotenv()
API_KEY = os.getenv('OPENFDA_API_KEY', '')
RECALL_URL = 'https://api.fda.gov/device/recall.json'

print(f'API Key: {"OK" if API_KEY else "FEHLT – Rate Limit aktiv (1k/Tag)"}')

## 1. MAUDE-Datensatz laden

In [ ]:
ROOT = Path('..') if Path('..').joinpath('data').exists() else Path('.')

df = pd.read_parquet(ROOT / 'data/raw/maude_pass1.parquet')
df['date_received'] = pd.to_datetime(df['date_received'], format='%Y%m%d', errors='coerce')
df['date_of_event'] = pd.to_datetime(df['date_of_event'], format='%Y%m%d', errors='coerce')

print(f'MAUDE Records: {len(df):,}')
print(f'Zeitraum: {df["date_received"].min().date()} → {df["date_received"].max().date()}')
print(f'Produktcodes: {df["product_code"].value_counts().to_dict()}')

## 2. Recalls aus FDA API ziehen

Wir ziehen Recalls für alle Produktcodes im Datensatz.
Recall-Klassen:
- **Class I** = höchstes Risiko (Lebensgefahr oder schwere Gesundheitsschäden)
- **Class II** = mittleres Risiko (temporäre oder reversible Schäden)
- **Class III** = geringstes Risiko (unwahrscheinliche Gesundheitsschäden)

In [ ]:
def fetch_recalls_for_product_code(product_code, api_key=''):
    """Alle Recalls für einen FDA Produktcode ziehen."""
    all_recalls = []
    skip = 0
    limit = 100

    while True:
        params = {
            'search': f'product_code:{product_code}',
            'limit': limit,
            'skip': skip
        }
        if api_key:
            params['api_key'] = api_key

        try:
            r = requests.get(RECALL_URL, params=params, timeout=30)
            if r.status_code == 404:
                break  # Keine Recalls für diesen Code
            r.raise_for_status()
            data = r.json()
            results = data.get('results', [])
            all_recalls.extend(results)

            total = data.get('meta', {}).get('results', {}).get('total', 0)
            skip += limit
            if skip >= total or not results:
                break
            time.sleep(0.3)
        except Exception as e:
            print(f'Fehler bei {product_code} (skip={skip}): {e}')
            break

    print(f'  {product_code}: {len(all_recalls)} Recalls gefunden')
    return all_recalls


# Alle Produktcodes im Datensatz
product_codes = df['product_code'].dropna().unique().tolist()
print(f'Ziehe Recalls für {len(product_codes)} Produktcodes: {product_codes}')

all_recalls = []
for code in product_codes:
    recalls = fetch_recalls_for_product_code(code, API_KEY)
    all_recalls.extend(recalls)

print(f'\nGesamt: {len(all_recalls)} Recall-Einträge')

## 3. Recall-Daten flachklopfen

In [ ]:
def flatten_recall(rec):
    return {
        'recall_number':       rec.get('recall_number', ''),
        'recalling_firm':      rec.get('recalling_firm', ''),
        'product_code':        rec.get('product_code', ''),
        'product_description': rec.get('product_description', ''),
        'reason_for_recall':   rec.get('reason_for_recall', ''),
        'recall_class':        rec.get('recall_class', ''),   # Class I / II / III
        'status':              rec.get('status', ''),
        'recall_initiation_date': rec.get('recall_initiation_date', ''),
        'event_date_terminated':  rec.get('event_date_terminated', ''),
        'distribution_pattern':   rec.get('distribution_pattern', ''),
    }

df_recalls = pd.DataFrame([flatten_recall(r) for r in all_recalls])
df_recalls['recall_initiation_date'] = pd.to_datetime(
    df_recalls['recall_initiation_date'], errors='coerce'
)

print(f'Shape: {df_recalls.shape}')
print('\nRecall-Klassen:')
print(df_recalls['recall_class'].value_counts())
print('\nTop recalling firms:')
print(df_recalls['recalling_firm'].value_counts().head(10))

# Speichern
df_recalls.to_parquet(ROOT / 'data/raw/recalls_raw.parquet', index=False)
print('\n→ data/raw/recalls_raw.parquet gespeichert')

## 4. Label-Strategie 1: Device-Level (`recalled_ever`)

Match über `product_code` + fuzzy `manufacturer_name`.
Einfach und robust – gut für Baseline-Modell.

In [ ]:
def normalize_firm(name):
    """Herstellernamen normalisieren für Fuzzy-Match."""
    if not isinstance(name, str):
        return ''
    return (
        name.lower()
        .replace('inc.', '').replace('inc', '')
        .replace('llc', '').replace('corp.', '').replace('corp', '')
        .replace('co.', '').replace('gmbh', '').replace('ag', '')
        .strip()
    )

df['manufacturer_norm'] = df['manufacturer_name'].apply(normalize_firm)
df_recalls['firm_norm'] = df_recalls['recalling_firm'].apply(normalize_firm)

# Alle (product_code, firm_norm) Paare aus Recall DB
recall_pairs = set(
    zip(df_recalls['product_code'], df_recalls['firm_norm'])
)

print(f'Eindeutige Recall-Paare (code + firm): {len(recall_pairs)}')

def has_recall_ever(row):
    code = row['product_code']
    firm = row['manufacturer_norm']
    # Exakter Match
    if (code, firm) in recall_pairs:
        return 1
    # Fuzzy Match (Threshold 85)
    for rc_code, rc_firm in recall_pairs:
        if rc_code == code and fuzz.ratio(firm, rc_firm) >= 85:
            return 1
    return 0

print('Berechne recalled_ever...')
df['recalled_ever'] = df.apply(has_recall_ever, axis=1)
print(f'\nLabel-Verteilung recalled_ever:')
print(df['recalled_ever'].value_counts())
print(f'Recall-Rate: {df["recalled_ever"].mean():.1%}')

## 5. Label-Strategie 2: Time-Window (`recalled_within_12m`)

Realistischeres Label: Gab es innerhalb von 12 Monaten **nach** dem MAUDE-Report
einen Recall für dasselbe Gerät?

Das simuliert echtes Frühwarnsystem-Verhalten.

In [ ]:
from datetime import timedelta

# Recall-Lookup: product_code → Liste von Recall-Daten
recall_dates_by_code = (
    df_recalls.dropna(subset=['recall_initiation_date'])
    .groupby('product_code')['recall_initiation_date']
    .apply(list)
    .to_dict()
)

def recalled_within_12m(row):
    code = row['product_code']
    report_date = row['date_received']
    if pd.isna(report_date) or code not in recall_dates_by_code:
        return 0
    window_end = report_date + timedelta(days=365)
    for rd in recall_dates_by_code[code]:
        if report_date <= rd <= window_end:
            return 1
    return 0

print('Berechne recalled_within_12m...')
df['recalled_within_12m'] = df.apply(recalled_within_12m, axis=1)
print(f'\nLabel-Verteilung recalled_within_12m:')
print(df['recalled_within_12m'].value_counts())
print(f'Recall-Rate: {df["recalled_within_12m"].mean():.1%}')

## 6. Datensatz prüfen und speichern

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Label-Verteilung über Zeit
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, label in zip(axes, ['recalled_ever', 'recalled_within_12m']):
    df.groupby(df['date_received'].dt.year)[label].mean().plot(
        kind='bar', ax=ax, color=['#e74c3c' if label == 'recalled_ever' else '#3498db']
    )
    ax.set_title(f'Recall-Rate per Jahr: {label}')
    ax.set_ylabel('Rate')
    ax.set_xlabel('Jahr')

plt.tight_layout()
plt.show()

print('\n=== Finale Datensatz-Info ===')
print(f'Records: {len(df):,}')
print(f'Features: {df.columns.tolist()}')
print(f'recalled_ever:     {df["recalled_ever"].sum():,} positiv ({df["recalled_ever"].mean():.1%})')
print(f'recalled_within_12m: {df["recalled_within_12m"].sum():,} positiv ({df["recalled_within_12m"].mean():.1%})')

In [ ]:
# Speichern
out_path = ROOT / 'data/processed/maude_labeled.parquet'
out_path.parent.mkdir(exist_ok=True)
df.to_parquet(out_path, index=False)
print(f'→ {out_path} gespeichert ({len(df):,} Records)')

# Kurze Qualitätsprüfung
df_check = pd.read_parquet(out_path)
assert len(df_check) == len(df), 'Record-Anzahl stimmt nicht!'
assert 'recalled_ever' in df_check.columns
assert 'recalled_within_12m' in df_check.columns
print('Qualitätsprüfung: OK')

## Nächste Schritte (Notebook 03)

1. Feature Engineering: Rolling-Window Complaint-Counts pro Gerät, Schweregrad-Score aus `event_type` + `patient_outcome`
2. Baseline-Modell: Logistic Regression auf `recalled_ever`
3. LightGBM auf `recalled_within_12m`
4. Evaluation: AUC-ROC, Precision-Recall (wegen Class-Imbalance wichtig)

**Hinweis Class Imbalance:** Recalls sind seltene Ereignisse – erwarte Recall-Rate < 10%.
→ Beim Modell `class_weight='balanced'` oder `scale_pos_weight` bei LightGBM setzen.